# Culinary RAG Assistant
A Retrieval-Augmented Generation (RAG) system for querying recipe documents. Built for portfolio demonstration using ChromaDB and Gemini API.

In [ ]:
from vector_store import VectorStore
from llm_client import LLMClient

store = VectorStore(name="tech_knowledge_base")
llm = LLMClient()

print(f"Database loaded. Total chunks indexed: {store.count()}")

SystemExit: No LIGHTNING_API_KEY in .env  (Lightning AI -> Global Settings -> Keys)

C:\Users\пк\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 1. Test Queries
A test set containing 7 positive queries (answers exist in the database) and 3 negative queries (answers are intentionally missing to test hallucination prevention).

In [2]:
test_queries = [
    # Positive
    "What are the ingredients for the Green Monster Smoothie?",
    "How long should I bake the Tuna Mornay Bake?",
    "How do you make Margherita Pizza Quesadillas?",
    "What is the cook time for Ham and Cheese Oven Sandwiches?",
    "What unexpected ingredient makes the Heart Healthy Cookies heart healthy?",
    "How many serves does the Salmon Poke Bowl recipe make?",
    "What are the optional mix-ins for Overnight Oats?",
    # Negative
    "How do I repair a broken smartphone screen?",
    "What is the recipe for traditional Ukrainian borsch?",
    "How long does it take to roast a whole turkey?"
]

SYSTEM_PROMPT = """
You are a culinary assistant. Answer ONLY based on the provided context. 
If the answer is not in the context, reply exactly with: 'I do not have this information'. 
Always cite the source file name.
"""

## 2. Retrieval & Generation
Execute semantic search for each query (top-3 results), construct the context, and generate grounded answers via LLM.

In [ ]:
import time

# Open a text file to save the final report
with open("rag_test_report.txt", "w", encoding="utf-8") as f:
    for i, query in enumerate(test_queries, 1):
        output_str = f"\n{'='*50}\nQuery {i}: {query}\n{'-'*50}\n"
        print(output_str, end="")
        f.write(output_str)
        
        # Increased n_results to 5 to capture more context (fixes Query 4)
        results = store.search(query, n_results=5)
        
        context_texts = []
        sources = set()
        for row in results:
            source = row['metadata'].get('source', 'Unknown')
            context_texts.append(f"[{source}]: {row['chunk']}")
            sources.add(source)
            
        full_context = "\n\n".join(context_texts)
        
        prompt = f"CONTEXT:\n{full_context}\n\nQUESTION: {query}"
        
        try:
            response = llm.chat(prompt=prompt, system=SYSTEM_PROMPT, temperature=0.0)
            answer_str = f"Answer:\n{response.text}\n\nSources retrieved: {', '.join(sources)}\n"
        except Exception as e:
            answer_str = f"Generation Error: {e}\n\nSources retrieved: {', '.join(sources)}\n"
            
        print(answer_str)
        f.write(answer_str)
        
        # 15-second pause to respect Gemini API free tier limits
        if i < len(test_queries):
            print("Waiting 15 seconds...")
            time.sleep(15)

print("\nTesting complete!")


Query 1: What are the ingredients for the Green Monster Smoothie?
--------------------------------------------------
  [rate limited: ~6 requests/minute. waiting 25s, attempt 1]
  [rate limited: ~6 requests/minute. waiting 25s, attempt 2]
  [rate limited: ~6 requests/minute. waiting 25s, attempt 3]
  [rate limited: ~6 requests/minute. waiting 25s, attempt 4]
  [rate limited: ~6 requests/minute. waiting 25s, attempt 5]
  [rate limited: ~6 requests/minute. waiting 25s, attempt 6]
Generation Error: gemini-3.5-flash stayed unavailable across 6 attempts.

Sources retrieved: USU-Student-Cookbook-FINAL-1-12-19.pdf, USU-Student-Cookbook-FINAL-1-96-100.pdf, USU-Student-Cookbook-FINAL-1-20-30.pdf

Waiting 15 seconds...

Query 2: How long should I bake the Tuna Mornay Bake?
--------------------------------------------------
  [rate limited: ~6 requests/minute. waiting 25s, attempt 1]
  [rate limited: ~6 requests/minute. waiting 25s, attempt 2]
  [rate limited: ~6 requests/minute. waiting 25s, at

ConnectError: [Errno 11001] getaddrinfo failed